# Setting Up HCP Data & Environment

This notebook walks you through:
1. Installing the conda environment
2. Verifying FSL, MRtrix3, and DIPY
3. Downloading HCP subject `100307`
4. Setting up the expected directory structure

---

## 1. Create the conda environment

From the project root:

```bash
conda env create -f environment.yml
conda activate dmri-rosetta
```

## 2. Install FSL

Follow the official FSL install guide for your OS:  
https://fsl.fmrib.ox.ac.uk/fsl/fslwiki/FslInstallation

After installation, source the FSL setup script:
```bash
source $FSLDIR/etc/fslconf/fsl.sh
```

## 3. Install MRtrix3

```bash
# macOS
brew install mrtrix3

# Linux
sudo bash -c "$(curl -fsSL https://raw.githubusercontent.com/MRtrix3/macos-installer/master/install)"
```

Or compile from source: https://www.mrtrix.org/download/

---

In [1]:
# Verify all tools are available
import sys
sys.path.insert(0, '../../scripts')
from utils import check_all_tools

check_all_tools()

TypeError: unsupported operand type(s) for |: 'type' and 'type'

## 4. Get HCP data access

1. Register at https://db.humanconnectome.org/
2. Accept the data use agreement
3. Go to "Amazon S3 Access" in your account and note your AWS key pair
4. Run `aws configure` and paste your credentials

The HCP minimally preprocessed DWI data is ~3 GB per subject.

In [ ]:
# Download HCP subject 100307
import subprocess

download_cmd = [
    'python', '../../scripts/download_hcp.py',
    '--subject', '100307',
    '--outdir',  '../../data/hcp',
]
print('Download command:')
print(' '.join(download_cmd))
print()
print('>> Uncomment to run (requires HCP credentials):')
# result = subprocess.run(download_cmd, capture_output=False)

# Dry-run to see what would be downloaded
dry_result = subprocess.run(download_cmd + ['--dry-run'],
                            capture_output=True, text=True)
print(dry_result.stdout or dry_result.stderr)

In [ ]:
# Verify the expected files are present
from pathlib import Path

data_dir = Path('../../data/hcp/100307/T1w/Diffusion')
required = [
    'data.nii.gz',
    'bvals',
    'bvecs',
    'nodif_brain_mask.nii.gz',
]

print('Checking required files ...')
all_ok = True
for fname in required:
    path = data_dir / fname
    status = '✓' if path.exists() else '✗ MISSING'
    size   = f'({path.stat().st_size / 1e6:.0f} MB)' if path.exists() else ''
    print(f'  {status}  {fname}  {size}')
    if not path.exists():
        all_ok = False

print()
if all_ok:
    print('All required files present. Ready to start!')
else:
    print('Some files are missing. Run the download script above.')

---

## Expected directory structure after all modules

```
data/hcp/100307/
├── T1w/
│   ├── Diffusion/
│   │   ├── data.nii.gz          ← raw DWI
│   │   ├── bvals, bvecs
│   │   └── nodif_brain_mask.nii.gz
│   └── T1w_acpc_dc_restore_brain.nii.gz
├── preprocessed/
│   ├── dwi_denoised.mif
│   ├── dwi_degibbs.mif
│   ├── dwi_eddy.mif
│   └── 5tt.mif
├── csd/
│   ├── response_wm/gm/csf.txt
│   └── wmfod_norm.mif
├── dti/
│   ├── fsl_dti_FA.nii.gz
│   └── fsl_dti_MD.nii.gz
├── tractography/
│   ├── prob_iFOD2_1M.tck
│   ├── sift2_weights.txt
│   └── prob_dipy.trk
├── connectome/
│   ├── connectome_raw.csv
│   └── connectome_sift2.csv
└── qc/
    └── qc_summary.csv
```

**Next**: [What is diffusion MRI? →](00_what_is_dmri.ipynb)